# NFXP — Empirisk applikation

**Oliver Ovdal Eiberg Jørgensen & Solveig Røndal-Liniger** — Dynamic Programming, Spring 2026

---

Estimerer den strukturelle model på Netto-scannerpanelet.  
Denne notebook starter med at estimere **kun $\alpha_2$** (brand 2-præference), mens alle øvrige parametre holdes fast.

**Parametre:**
$$\theta = [\alpha_0,\; \alpha_2,\; \gamma,\; \beta^{sc}_{12},\; \beta^{sc}_{21},\; \beta^{dep}_1,\; \beta^{dep}_2]$$

- $\alpha_1 = 0$ er normaliseringen
- $\alpha_0$ er no-purchase-konstantleddet (ekstra parameter i empirisk model)

## 1. Import

In [ ]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.stats import norm
import matplotlib.pyplot as plt

## 2. Model-dimensioner og konstanter

In [ ]:
J         = 2       # number of brands
D_MAX     = 48      # duration cap (weeks since last purchase)
DELTA     = 0.95    # discount factor
N_CHOICES = J + 1   # {0=no purchase, 1=brand 1, 2=brand 2}

EMPIRICAL_BRANDS        = [1, 2]
EMPIRICAL_PRICE_COLUMNS = {1: "Brand_1_price", 2: "Brand_2_price"}

VFI_TOL      = 1e-10
VFI_MAXITER  = 2_000
TRANS_SMOOTH = 1e-3   # Laplace smoothing for transition matrix

# All 2^J joint promotion states as binary vectors
promo_states = np.array(
    [[(s >> j) & 1 for j in range(J)] for s in range(2 ** J)], dtype=int
)
N_PROMO = len(promo_states)

print(f"J={J}, D_MAX={D_MAX}, N_CHOICES={N_CHOICES}, N_PROMO={N_PROMO}")
print("promo_states:\n", promo_states)

## 3. Indlæs data

> **Ret `PRICE_PROMO_DATA`-stien** til din lokale kopi.

In [ ]:
# ── Data paths — update these ─────────────────────────────────────────────────
SCANNER_DATA     = "/Users/solveigroendalliniger/Library/Mobile Documents/com~apple~CloudDocs/Personlig/netto_df.csv"
PRICE_PROMO_DATA = "INDSÆT_STI_HER/Price_Promo.xlsx"   # <-- update this path
# ─────────────────────────────────────────────────────────────────────────────

# Load scanner panel
netto_df = pd.read_csv(SCANNER_DATA)
netto_df = netto_df[netto_df["week_num"] != 53].copy()
netto_df["Brand"]       = netto_df["Brand"].astype(str)
netto_df["Price"]       = pd.to_numeric(netto_df["Price"],       errors="coerce").round(1)
netto_df["Customer_ID"] = pd.to_numeric(netto_df["Customer_ID"], errors="coerce").astype("Int64")
netto_df["week_num"]    = pd.to_numeric(netto_df["week_num"],    errors="coerce").astype("Int64")
netto_df["brand_id"]    = pd.to_numeric(netto_df["Brand"],       errors="coerce").fillna(0).astype(int)

# Load price/promo data from third Excel sheet
wb             = pd.ExcelFile(PRICE_PROMO_DATA)
price_promo_df = pd.read_excel(wb, sheet_name=wb.sheet_names[2])

req_cols = [
    "WeekNum",
    "RRP_Brand_1", "Discount_Brand_1", "Promo_Brand_1",
    "RRP_Brand_2", "Discount_Brand_2", "Promo_Brand_2",
]
price_promo_df = price_promo_df[req_cols].copy()
for col in req_cols:
    price_promo_df[col] = pd.to_numeric(price_promo_df[col], errors="coerce")
price_promo_df = price_promo_df.dropna(subset=["WeekNum"]).copy()
price_promo_df["WeekNum"] = price_promo_df["WeekNum"].astype(int)

# Binary price: RRP when not on promotion, mean discount price when on promotion
for brand in EMPIRICAL_BRANDS:
    promo_col = f"Promo_Brand_{brand}"
    disc_col  = f"Discount_Brand_{brand}"
    rrp_col   = f"RRP_Brand_{brand}"
    price_col = EMPIRICAL_PRICE_COLUMNS[brand]
    mean_col  = f"Mean_Discount_Brand_{brand}"

    price_promo_df[promo_col] = price_promo_df[promo_col].fillna(0).astype(int)
    mask      = (price_promo_df[promo_col] == 1) & price_promo_df[disc_col].notna()
    mean_disc = price_promo_df.loc[mask, disc_col].mean()
    if not np.isfinite(mean_disc):
        mean_disc = price_promo_df[disc_col].mean()
    if not np.isfinite(mean_disc):
        mean_disc = price_promo_df[rrp_col].median()

    price_promo_df[mean_col]  = mean_disc
    price_promo_df[price_col] = np.where(
        price_promo_df[promo_col] == 1, mean_disc, price_promo_df[rrp_col]
    )

# Merge weekly prices onto scanner panel
merge_cols = [
    "WeekNum",
    "RRP_Brand_1", "Discount_Brand_1", "Mean_Discount_Brand_1", "Promo_Brand_1", "Brand_1_price",
    "RRP_Brand_2", "Discount_Brand_2", "Mean_Discount_Brand_2", "Promo_Brand_2", "Brand_2_price",
]
netto_df = netto_df.merge(
    price_promo_df[merge_cols],
    left_on="week_num", right_on="WeekNum",
    how="left", validate="many_to_one",
)
netto_df = netto_df.loc[
    netto_df["Brand_1_price"].notna() & netto_df["Brand_2_price"].notna()
].copy()

print(f"Panel: {netto_df['Customer_ID'].nunique():,} households, {netto_df['week_num'].nunique()} weeks, {len(netto_df):,} rows")

## 4. Empirisk prisproces

Estimér den ugentlige overgangsmatrix og median-priser per kampagnetilstand.

In [ ]:
def empirical_promo_idx(frame: pd.DataFrame) -> np.ndarray:
    """Map Promo_Brand_1/2 columns to index in promo_states."""
    mat = frame[[f"Promo_Brand_{b}" for b in EMPIRICAL_BRANDS]].fillna(0).astype(int).to_numpy()
    return mat @ (2 ** np.arange(J))


def build_weekly_state_table(data: pd.DataFrame) -> pd.DataFrame:
    cols   = ["week_num", "Brand_1_price", "Brand_2_price", "Promo_Brand_1", "Promo_Brand_2"]
    weekly = data[cols].dropna().drop_duplicates().sort_values("week_num").reset_index(drop=True)
    weekly["promo_idx"] = empirical_promo_idx(weekly)
    return weekly


def estimate_promo_transition(weekly: pd.DataFrame, smoothing: float = TRANS_SMOOTH) -> np.ndarray:
    """Estimate weekly Markov transition matrix over promotion states."""
    idx    = weekly["promo_idx"].to_numpy(dtype=int)
    counts = np.full((N_PROMO, N_PROMO), smoothing)
    np.add.at(counts, (idx[:-1], idx[1:]), 1.0)
    return counts / counts.sum(axis=1, keepdims=True)


def build_price_by_promo(weekly: pd.DataFrame) -> np.ndarray:
    """Median offered price per joint promotion state, with fallbacks for unobserved states."""
    pbp = np.full((N_PROMO, J), np.nan)
    for s in range(N_PROMO):
        rows = weekly.loc[weekly["promo_idx"] == s]
        if len(rows) == 0:
            continue
        for b_idx, b in enumerate(EMPIRICAL_BRANDS):
            pbp[s, b_idx] = rows[EMPIRICAL_PRICE_COLUMNS[b]].median()
    # Fallback: use own-promotion status to fill unobserved joint states
    for s, pvec in enumerate(promo_states):
        for b_idx, b in enumerate(EMPIRICAL_BRANDS):
            if np.isfinite(pbp[s, b_idx]):
                continue
            fb = weekly.loc[
                weekly[f"Promo_Brand_{b}"].astype(int) == int(pvec[b_idx]),
                EMPIRICAL_PRICE_COLUMNS[b],
            ].median()
            pbp[s, b_idx] = fb if np.isfinite(fb) else weekly[EMPIRICAL_PRICE_COLUMNS[b]].median()
    return pbp


weekly_state   = build_weekly_state_table(netto_df)
PROMO_TRANS    = estimate_promo_transition(weekly_state)
PRICE_BY_PROMO = build_price_by_promo(weekly_state)

lbl = [f"e={tuple(int(v) for v in r)}" for r in promo_states]
print("Median prices by promotion state:")
print(pd.DataFrame(PRICE_BY_PROMO, index=lbl, columns=["Brand 1", "Brand 2"]).to_string(float_format=lambda x: f"{x:.3f}"))
print("\nEstimated weekly promotion transition matrix:")
print(pd.DataFrame(PROMO_TRANS, index=lbl, columns=lbl).to_string(float_format=lambda x: f"{x:.3f}"))

## 5. Panelforberedelse

Rekonstruér $( \ell_{it}, d_{it}, e_t )$ og aggregér til state-choice counts.

In [ ]:
def prepare_panel(data: pd.DataFrame, d_max: int = D_MAX):
    panel = data.sort_values(["Customer_ID", "week_num"]).copy()
    panel["choice"]         = panel["brand_id"].where(panel["brand_id"].isin(EMPIRICAL_BRANDS), 0).astype(int)
    panel["purchase_brand"] = panel["choice"].where(panel["choice"] > 0)
    panel["purchase_week"]  = panel["week_num"].where(panel["choice"] > 0)

    grp = panel.groupby("Customer_ID", sort=False)
    panel["last_brand_incl"] = grp["purchase_brand"].ffill()
    panel["last_week_incl"]  = grp["purchase_week"].ffill()
    panel["pre_last_brand"]  = grp["last_brand_incl"].shift(1)   # last brand at start of period
    panel["pre_last_week"]   = grp["last_week_incl"].shift(1)    # week of last purchase
    panel["pre_dur_weeks"]   = panel["week_num"] - panel["pre_last_week"]

    # Keep only rows where the state (ℓ, d, e) is fully observed
    usable = panel.loc[
        panel["pre_last_brand"].isin(EMPIRICAL_BRANDS)
        & panel["pre_dur_weeks"].notna()
        & panel["Promo_Brand_1"].notna()
        & panel["Promo_Brand_2"].notna()
    ].copy()

    usable["Y"]     = usable["choice"].astype(int)
    usable["L"]     = usable["pre_last_brand"].astype(int)
    usable["D"]     = np.clip(usable["pre_dur_weeks"].to_numpy(dtype=float) - 1.0, 0, d_max).astype(int)
    usable["E_IDX"] = empirical_promo_idx(usable)

    # Aggregate to state-choice counts for fast likelihood evaluation
    counts = np.zeros((J, d_max + 1, N_PROMO, N_CHOICES), dtype=float)
    np.add.at(
        counts,
        (usable["L"].to_numpy() - 1, usable["D"].to_numpy(),
         usable["E_IDX"].to_numpy(), usable["Y"].to_numpy()),
        1.0,
    )
    return usable, counts


panel_df, OBS_COUNTS = prepare_panel(netto_df)
N_OBS = int(OBS_COUNTS.sum())

shares = OBS_COUNTS.sum(axis=(0, 1, 2)) / N_OBS
print(f"NFXP observations: {N_OBS:,}")
print(f"  No purchase : {shares[0]:.1%}")
for j in range(1, J + 1):
    print(f"  Brand {j}     : {shares[j]:.1%}")
print(f"  Duration cap: {D_MAX}  |  Capped: {(panel_df['D'] == D_MAX).mean():.1%}")

## 6. Vectoriseret empirisk VFI og CCPs

Samme struktur som Monte Carlo-notebooken, men med estimeret `PROMO_TRANS`, observerede `PRICE_BY_PROMO` og $\alpha_0$ i no-purchase-nytten:

$$Q_0(\ell, d, e) = \alpha_0 + \alpha_\ell - \beta^{dep}_\ell \cdot d + \delta \cdot EV(\ell, d{+}1, e')$$
$$Q_j(\ell, d, e) = \alpha_j - \gamma \cdot p_j(e) - \beta^{sc}_{\ell j} + \delta \cdot EV(j, 1, e')$$

In [ ]:
def solve_vfi(
    alpha_0: float,
    alpha: np.ndarray,
    gamma: float,
    beta_sc: np.ndarray,
    beta_dep: np.ndarray,
    tol: float = VFI_TOL,
    max_iter: int = VFI_MAXITER,
) -> np.ndarray:
    """Vectorised VFI (inner loop). V shape: (J, D_MAX+1, N_PROMO)."""
    dur_idx  = np.arange(D_MAX + 1)
    duration = dur_idx + 1.0          # paper-d = d_idx + 1
    next_dur = np.minimum(dur_idx + 1, D_MAX)

    # No-purchase flow utility: (J, D_MAX+1)
    no_purch = alpha_0 + alpha[:, None] - beta_dep[:, None] * duration[None, :]

    Q = np.empty((J, D_MAX + 1, N_PROMO, N_CHOICES))
    V = np.zeros((J, D_MAX + 1, N_PROMO))

    for _ in range(max_iter):
        # Integrate V over next promotion state
        EV = (V.reshape(J * (D_MAX + 1), N_PROMO) @ PROMO_TRANS.T).reshape(J, D_MAX + 1, N_PROMO)

        # No purchase: next state is (l, min(d+1, D_MAX))
        Q[..., 0] = no_purch[:, :, None] + DELTA * EV[:, next_dur, :]

        # Purchase brand j: next state is (j, d_idx=0)
        for j in range(J):
            Q[..., j + 1] = (
                alpha[j]
                - gamma * PRICE_BY_PROMO[:, j][None, None, :]   # (1, 1, N_PROMO)
                - beta_sc[:, j][:, None, None]                  # (J, 1, 1)
                + DELTA * EV[j, 0, :][None, None, :]            # (1, 1, N_PROMO)
            )

        # Log-sum-exp (Emax under iid Type-I extreme value shocks)
        q_max = Q.max(axis=3)
        V_new = q_max + np.log(np.exp(Q - q_max[..., None]).sum(axis=3))
        if np.max(np.abs(V_new - V)) < tol:
            return V_new
        V = V_new

    return V


def compute_ccps(
    V: np.ndarray,
    alpha_0: float,
    alpha: np.ndarray,
    gamma: float,
    beta_sc: np.ndarray,
    beta_dep: np.ndarray,
) -> np.ndarray:
    """Conditional choice probabilities from converged V. P shape: (J, D_MAX+1, N_PROMO, N_CHOICES)."""
    dur_idx  = np.arange(D_MAX + 1)
    duration = dur_idx + 1.0
    next_dur = np.minimum(dur_idx + 1, D_MAX)

    no_purch = alpha_0 + alpha[:, None] - beta_dep[:, None] * duration[None, :]
    EV = (V.reshape(J * (D_MAX + 1), N_PROMO) @ PROMO_TRANS.T).reshape(J, D_MAX + 1, N_PROMO)

    Q = np.empty((J, D_MAX + 1, N_PROMO, N_CHOICES))
    Q[..., 0] = no_purch[:, :, None] + DELTA * EV[:, next_dur, :]
    for j in range(J):
        Q[..., j + 1] = (
            alpha[j]
            - gamma * PRICE_BY_PROMO[:, j][None, None, :]
            - beta_sc[:, j][:, None, None]
            + DELTA * EV[j, 0, :][None, None, :]
        )

    w = np.exp(Q - Q.max(axis=3, keepdims=True))
    return w / w.sum(axis=3, keepdims=True)


def log_lik(counts: np.ndarray, P: np.ndarray) -> float:
    return float(np.sum(counts * np.log(np.maximum(P, 1e-300))))


print("VFI and CCP functions defined.")

## 7. Fastlagte parametre og 1D-objective

Alle parametre undtagen $\alpha_2$ holdes fast. Tilpas disse efter behov.

In [ ]:
# ── Fixed parameters (all except alpha_2) ────────────────────────────────────
FIXED_ALPHA_0  = 2.00
FIXED_GAMMA    = 0.05
FIXED_BETA_SC  = np.array([[0.00, 0.25], [0.25, 0.00]])
FIXED_BETA_DEP = np.array([0.275, 0.275])
# ─────────────────────────────────────────────────────────────────────────────


def neg_ll_alpha2(alpha_2: float) -> float:
    """NFXP objective: solve VFI (inner loop), then return negative log-likelihood."""
    alpha = np.array([0.0, alpha_2])   # alpha_1 = 0 is the normalisation
    V = solve_vfi(FIXED_ALPHA_0, alpha, FIXED_GAMMA, FIXED_BETA_SC, FIXED_BETA_DEP)
    P = compute_ccps(V, FIXED_ALPHA_0, alpha, FIXED_GAMMA, FIXED_BETA_SC, FIXED_BETA_DEP)
    return -log_lik(OBS_COUNTS, P)


print("Fixed parameters:")
print(f"  alpha_0   = {FIXED_ALPHA_0}")
print(f"  alpha_1   = 0.0  (normalisation)")
print(f"  gamma     = {FIXED_GAMMA}")
print(f"  beta_sc   = {FIXED_BETA_SC}")
print(f"  beta_dep  = {FIXED_BETA_DEP}")

## 8. Grid search over $\alpha_2$

Kortlæg likelihood-overfladen og find et godt startpunkt.

In [ ]:
GRID = np.linspace(-1.0, 3.0, 25)

t0      = time.perf_counter()
ll_grid = []
for i, a2 in enumerate(GRID):
    nll = neg_ll_alpha2(float(a2))
    ll_grid.append(-nll)
    if (i + 1) % 5 == 0:
        print(f"  Grid {i+1:>2}/{len(GRID)}  alpha_2={a2:+.3f}  LL={-nll:.1f}")

ll_grid       = np.array(ll_grid)
best_grid_idx = int(np.argmax(ll_grid))
print(f"\nBest grid point: alpha_2 = {GRID[best_grid_idx]:.3f}  (LL = {ll_grid[best_grid_idx]:.1f})")
print(f"Grid search: {time.perf_counter() - t0:.1f}s")

## 9. Finpudsning med `minimize_scalar`

In [ ]:
# Search within two grid steps of the best grid point
a2_lo = float(GRID[max(best_grid_idx - 2, 0)])
a2_hi = float(GRID[min(best_grid_idx + 2, len(GRID) - 1)])

t0  = time.perf_counter()
res = minimize_scalar(
    neg_ll_alpha2,
    bounds=(a2_lo, a2_hi),
    method="bounded",
    options={"xatol": 1e-6, "maxiter": 200},
)
elapsed = time.perf_counter() - t0

ALPHA_2_HAT = float(res.x)
NLL_HAT     = float(res.fun)   # negative log-likelihood at optimum
LL_HAT      = -NLL_HAT

print(f"Converged: {res.success}  |  Evaluations: {res.nfev}  |  Time: {elapsed:.1f}s")
print(f"\n{'='*40}")
print(f"  alpha_2_hat = {ALPHA_2_HAT:.6f}")
print(f"  Log-lik     = {LL_HAT:.2f}")
print(f"{'='*40}")

## 10. Standardfejl og p-værdi for $\alpha_2$

Standardfejlen beregnes fra den numeriske andenafledning af den negative log-likelihood ved MLE-estimatet:

$$\widehat{\text{Var}}(\hat{\alpha}_2) = \left[\frac{\partial^2 (-\ell)}{\partial \alpha_2^2}\Bigg|_{\hat{\alpha}_2}\right]^{-1}$$

Andenafledningen approksimeres med centraldifferens. P-værdien er for tosidet test $H_0: \alpha_2 = 0$.

In [ ]:
# Step size for numerical second derivative (scales with estimate magnitude)
h = max(1e-4, abs(ALPHA_2_HAT) * 1e-4)

# Central-difference approximation of d²(-LL)/dα₂²
d2_nll = (
    neg_ll_alpha2(ALPHA_2_HAT + h)
    - 2.0 * NLL_HAT
    + neg_ll_alpha2(ALPHA_2_HAT - h)
) / h ** 2

# Inverse Hessian gives asymptotic variance of the MLE
if d2_nll > 0:
    se_alpha2 = np.sqrt(1.0 / d2_nll)
else:
    se_alpha2 = np.nan
    print("Warning: non-positive second derivative — SE not identified.")

# Two-sided Wald test: H0: alpha_2 = 0
z_stat  = ALPHA_2_HAT / se_alpha2
p_value = float(2 * norm.sf(abs(z_stat)))

print(f"{'='*50}")
print(f"  alpha_2_hat = {ALPHA_2_HAT:+.6f}")
print(f"  Std. error  = {se_alpha2:.6f}")
print(f"  z-statistic = {z_stat:.4f}")
print(f"  p-value     = {p_value:.4e}   (H0: alpha_2 = 0)")
print(f"{'='*50}")

In [ ]:
print(f"p-value (H0: alpha_2 = 0):  {p_value:.4e}")

## 11. Resultater

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(GRID, ll_grid, "o-", color="steelblue", lw=1.8, ms=5, label="Grid LL")
ax.axvline(ALPHA_2_HAT, color="firebrick", lw=2, ls="--",
           label=rf"$\hat{{\alpha}}_2 = {ALPHA_2_HAT:.4f}$")
# Confidence interval shading (±1.96 SE)
if np.isfinite(se_alpha2):
    ax.axvspan(ALPHA_2_HAT - 1.96 * se_alpha2, ALPHA_2_HAT + 1.96 * se_alpha2,
               alpha=0.15, color="firebrick", label="95% CI")
ax.set_xlabel(r"$\alpha_2$", fontsize=12)
ax.set_ylabel("Log-likelihood", fontsize=11)
ax.set_title(r"Empirisk log-likelihood som funktion af $\alpha_2$", fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("nfxp_empirical_alpha2.pdf", bbox_inches="tight")
plt.show()

# Observed vs. predicted choice shares at the estimate
alpha_hat    = np.array([0.0, ALPHA_2_HAT])
V_hat        = solve_vfi(FIXED_ALPHA_0, alpha_hat, FIXED_GAMMA, FIXED_BETA_SC, FIXED_BETA_DEP)
P_hat        = compute_ccps(V_hat, FIXED_ALPHA_0, alpha_hat, FIXED_GAMMA, FIXED_BETA_SC, FIXED_BETA_DEP)
state_counts = OBS_COUNTS.sum(axis=3)
obs_shares   = OBS_COUNTS.sum(axis=(0, 1, 2)) / N_OBS
pred_shares  = (state_counts[..., None] * P_hat).sum(axis=(0, 1, 2)) / N_OBS

fit_table = pd.DataFrame({
    "Choice":           ["No purchase", "Brand 1", "Brand 2"],
    "Observed share":   obs_shares,
    "Predicted share":  pred_shares,
    "Difference":       pred_shares - obs_shares,
})
print("Observed vs. predicted choice shares:")
print(fit_table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))